# 🚀 SUSA-1B CodeMind — Master Google Colab GPU Pipeline

> **An original coding-focused Transformer language model built and trained from scratch under the SUSA technology ecosystem.**

---

### 📌 Zero-Cost Engineering Principles:
- **Hardware Cost:** ₹0 (leveraging Google Colab Free Tier GPUs — NVIDIA T4 / V100).
- **Core Model:** Real decoder-only Transformer with RoPE, SwiGLU, RMSNorm, GQA, and tied embeddings.
- **Persistent Checkpoints:** High-speed NVMe writes locally synced continuously to Google Drive for disconnection recovery.
- **Full Stack:** Pretraining, Instruction Tuning, Sandboxed Evaluation, and Inference.

## Step 1: Connect to GitHub & Clone Repository

In [ ]:
import os
import sys

# Enter your repository URL (default connects to SUSA-1B CodeMind)
REPO_URL = input("Enter your GitHub repository URL: ").strip() or "https://github.com/saketh-nandu/susa-1b-codemind.git"
REPO_NAME = "susa-1b-codemind"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest approved commit in {REPO_NAME}...")
    !cd {REPO_NAME} && git pull origin main
else:
    print(f"Cloning {REPO_URL} into Colab runtime...")
    !git clone {REPO_URL} {REPO_NAME}

%cd {REPO_NAME}
!pwd && git log -n 1 --oneline

## Step 2: Install Dependencies

In [ ]:
# Install project in editable mode with colab and dev dependencies
!pip install --upgrade pip
!pip install -e ".[colab,dev]"
!pip install psutil pyyaml tokenizers tqdm

## Step 3: Inspect GPU Hardware & Environment Diagnostics

In [ ]:
import torch
from susacodemind.utils.system_checker import SystemInfo

sys_info = SystemInfo.inspect()
print(sys_info.summary())

if not sys_info.cuda_available:
    print("\n" + "!" * 68)
    print("  [CRITICAL WARNING] No CUDA GPU detected in this Colab session!")
    print("  Please change runtime accelerator:")
    print("  1. Go to top menu: Runtime -> Change runtime type")
    print("  2. Hardware accelerator: Select 'T4 GPU'")
    print("  3. Save and re-run this notebook.")
    print("!" * 68)
    raise SystemExit("Execution paused: GPU required for training beyond smoke test.")
else:
    print(f"\n[OK] CUDA GPU ready: {torch.cuda.get_device_name(0)} ({sys_info.gpu_devices[0]['total_vram_gb']} GiB VRAM)")

## Step 4: Mount Google Drive (Persistent Checkpoint Storage)

In [ ]:
# Authorize Google Drive via standard Colab flow
ENABLE_DRIVE_SYNC = True #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/SUSA_CodeMind_Checkpoints"

if ENABLE_DRIVE_SYNC:
    from google.colab import drive
    print("Authorizing Google Drive mount...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive persistent directory ready at: {DRIVE_DIR}")
else:
    print("Drive synchronization disabled. Checkpoints will remain in local runtime.")

## Step 5: Select Architecture Profile & Run Feasibility Check

In [ ]:
# Select model configuration profile
PROFILE = "small" #@param ["smoke", "small", "medium", "target_1b"]

from susacodemind.model.config import ModelConfig
from susacodemind.utils.resource_estimator import estimate_resources

cfg_path = f"configs/{PROFILE}.yaml"
model_cfg = ModelConfig.from_yaml(cfg_path)
print(model_cfg.print_parameter_summary())

# Check memory feasibility against Colab VRAM
res = estimate_resources(model_cfg, batch_size=2, seq_len=model_cfg.max_seq_len)
print(res.summary())

if PROFILE == "target_1b":
    print("\n[NOTICE] Profile 'target_1b' (~1B parameters) requires gradient checkpointing and micro-batching on free T4.")
    print("For rapid initial experiments, 'small' (~34M) or 'medium' (~222M) is recommended.")

## Step 6: Run Comprehensive Automated Test Suite

In [ ]:
# Verify all 49 unit tests pass on this runtime
!pytest tests/ -v

## Step 7: Train Tokenizer

In [ ]:
TOKENIZER_DIR = f"tokenizer_cache/{PROFILE}"
VOCAB_SIZE = model_cfg.vocab_size

# Train or load Byte-Level BPE tokenizer with FIM tokens
if not os.path.exists(os.path.join(TOKENIZER_DIR, "tokenizer.json")):
    print(f"Training tokenizer for profile {PROFILE} with vocab size {VOCAB_SIZE:,}...")
    !python scripts/train_tokenizer.py --use-synthetic --vocab-size {VOCAB_SIZE} --output-dir {TOKENIZER_DIR}
else:
    print(f"Tokenizer already exists at {TOKENIZER_DIR}.")

## Step 8: Prepare Sharded Binary Dataset

In [ ]:
# Clean, deduplicate, scrub secrets, and write binary shards to data/
if not os.path.exists("data/train.bin"):
    print("Cleaning, scrubbing secrets, and creating binary dataset shards...")
    !python scripts/prepare_dataset.py --tokenizer-dir {TOKENIZER_DIR} --use-synthetic --output-dir data
else:
    print("Binary dataset shards already exist in data/")

## Step 9: Launch Pretraining with Periodic Checkpointing

In [ ]:
PRETRAIN_STEPS = 50 #@param {type:"integer"}
RESUME_CHECKPOINT = "" #@param {type:"string"}

train_cmd = f"python scripts/train.py --config configs/{PROFILE}.yaml --train-bin data/train.bin --val-bin data/val.bin --max-steps {PRETRAIN_STEPS}"
if RESUME_CHECKPOINT.strip():
    train_cmd += f" --resume-from {RESUME_CHECKPOINT.strip()}"

print(f"Executing pretraining: {train_cmd}")
!{train_cmd}

## Step 10: Run Supervised Instruction Fine-Tuning

In [ ]:
import glob
available_ckpts = sorted(glob.glob("checkpoints/*.pt"))
base_ckpt = available_ckpts[-1] if available_ckpts else ""

if base_ckpt:
    print(f"Instruction-tuning base checkpoint: {base_ckpt}...")
    !python scripts/train_instruct.py --base-checkpoint {base_ckpt} --tokenizer-dir {TOKENIZER_DIR} --use-synthetic --max-steps 20 --output-dir checkpoints/instruct
else:
    print("No base checkpoint available to instruction-tune. Please run Step 9 first.")

## Step 11: Run Benchmark Evaluation & Perplexity

In [ ]:
eval_ckpt = "checkpoints/instruct/checkpoint_instruct_final.pt"
if not os.path.exists(eval_ckpt):
    eval_ckpt = base_ckpt

if os.path.exists(eval_ckpt):
    print(f"Evaluating checkpoint: {eval_ckpt}...")
    !python scripts/evaluate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --val-bin data/val.bin
else:
    print("No checkpoint available for evaluation.")

## Step 12: Test Interactive Generation

In [ ]:
if os.path.exists(eval_ckpt):
    PROMPT_TEXT = "def quicksort(arr: list[int]) -> list[int]:\n" #@param {type:"string"}
    !python scripts/generate.py --checkpoint {eval_ckpt} --tokenizer-dir {TOKENIZER_DIR} --prompt "{PROMPT_TEXT}" --max-new-tokens 48 --temperature 0.7


## Step 13: Sync Checkpoints to Google Drive & Export Run Manifest

In [ ]:
if ENABLE_DRIVE_SYNC and os.path.exists("/content/drive/MyDrive"):
    print(f"Syncing checkpoints to Google Drive: {DRIVE_DIR}...")
    !python scripts/sync_checkpoints.py --src-dir checkpoints --dest-dir "{DRIVE_DIR}"
    if os.path.exists("checkpoints/instruct"):
        !python scripts/sync_checkpoints.py --src-dir checkpoints/instruct --dest-dir "{DRIVE_DIR}/instruct"
else:
    print("Skipping Drive synchronization (Drive not mounted).")

## Step 14: Execution Summary

In [ ]:
saved_ckpts = sorted(glob.glob("checkpoints/**/*.pt", recursive=True))
print("=" * 70)
print("  SUSA-1B CODEMIND — COMPLETE RUN SUMMARY")
print("=" * 70)
print(f"  Profile:              {PROFILE}")
print(f"  Parameters:           {model_cfg.calculate_exact_parameters()['formatted_total']}")
print(f"  Saved Checkpoints:    {len(saved_ckpts)}")
for c in saved_ckpts:
    print(f"    - {c}")
if ENABLE_DRIVE_SYNC:
    print(f"  Google Drive Backup:  {DRIVE_DIR}")
print("  Total Compute Cost:   ₹0 (Free Google Colab Session)")
print("=" * 70)
print("All SUSA-1B CodeMind milestones executed successfully!")